# M14 rerun with a 60 km/h ramp (seed 0) on Google Colab

This notebook runs the complete M14 study from `m14/`: the scenario screening (E0) and capacity check, the initial
SUMO dataset, the 5-member DeepONet ensemble, surrogate-trained PPO with data aggregation (up to 5 rounds),
direct SUMO-PPO at 1000 episodes, the ALINEA / PI-ALINEA / constant baselines, the final evaluation on the
ID and OOD sets (including Surrogate-MPC), and the paper's Tables I and II.

**Everything is written to Google Drive** (`WORK` below), so a lost session costs at most the work in progress:
rerun cells 1-3 and then cell 6. Finished stages are reused; the direct PPO run continues from its latest
checkpoint (every 80 episodes); an interrupted aggregation round or evaluation is moved aside (never deleted)
and redone.

**Before the first run**
1. The code comes from GitHub (`REPO_URL`, `BRANCH`): the commit you run must contain the 60 km/h `m14/` changes.
2. Runtime: SUMO is CPU-only, so the number of CPU cores sets the pace (cell 3 prints it). A GPU only speeds up
   DeepONet training. Background execution (Colab Pro+) lets the run continue with the browser closed; otherwise
   keep the tab open.

**Compute (from the 120 km/h study)**, in CPU core-hours: E0 ~2.7, initial dataset ~2.0, DeepONet ensemble ~10
(much less on a GPU), aggregation loop ~5 (3.3 h of PPO plus SUMO validation), direct PPO ~9 on **one** core (the
longest single job), baselines ~5 (widened ALINEA grid, 540 episodes, plus the constant sweep), final evaluation ~11 plus ~9 for Surrogate-MPC. With 8 cores expect roughly
14-16 h of wall time; with 2 cores several days.

| cell | what | when |
|---|---|---|
| 1 | parameters | every session |
| 2 | Drive + code | every session |
| 3 | install SUMO + check | every session |
| 4 | one SUMO episode on the 60 km/h road | first session |
| 5 | **E0 + capacity check** (decision point) | first session |
| 6 | launch / resume the pipeline in the background | after cell 5, and after every lost session |
| 7 | status | any time |
| 8 | tables | at the end |
| 9 | archive of the results on Drive | at the end |

In [ ]:
# 1. Parameters
REPO_URL = "https://github.com/LejunZhou/traffic-surrogate-rl.git"
BRANCH = "main"
WORK = "/content/drive/MyDrive/m14_ramp60"      # project copy + every output (persistent across sessions)
SEED = 0
ROUNDS = 5                                       # aggregation rounds (stop rule: two rounds improving < 2)
BUDGETS = [1000]                                 # direct SUMO-PPO budgets in episodes
ALINEA_GRID = {}                                 # {} = tuner default (the widened grid, 540 episodes):
                                                 #   {"dets": [12, 13, 14, 15], "rhos": [20, 23, 26, 30, 34],
                                                 #    "kis": [10, 20, 35], "kps": [2, 4, 8]}
                                                 # Fix it before the first launch of cell 6: tuning is done once and reused.
UPDATE_CODE = False                              # True: refresh the code in WORK from GitHub (keeps data/ runs/ reports/).
                                                 # Do not change code in the middle of a study unless you mean to.

In [ ]:
# 2. Drive + code
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, subprocess, datetime

CLONE = "/content/traffic-surrogate-rl"
if not os.path.isdir(CLONE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CLONE], check=True)
else:
    subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=True)
commit = subprocess.run(["git", "-C", CLONE, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

CODE_ITEMS = ["src", "scripts", "configs", "tests", "docs", "run.py", "pyproject.toml", "README.md"]
first_copy = not os.path.isdir(WORK)
if first_copy:
    shutil.copytree(f"{CLONE}/m14", WORK, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
elif UPDATE_CODE:
    for item in CODE_ITEMS:
        src, dst = f"{CLONE}/m14/{item}", f"{WORK}/{item}"
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
        else:
            shutil.copy2(src, dst)
if first_copy or UPDATE_CODE:
    with open(f"{WORK}/CODE_VERSION.txt", "a") as f:
        f.write(f"{datetime.datetime.now().isoformat(timespec='seconds')}  {commit}  {'first copy' if first_copy else 'update'}\n")
print(open(f"{WORK}/CODE_VERSION.txt").read())
os.chdir(WORK)
!grep -n "ramp_speed_limit_mps" configs/scenario.yaml

In [ ]:
# 3. Install SUMO and dependencies, check the runtime
!pip install -q "eclipse-sumo==1.27.1" "traci==1.27.1" "sumolib==1.27.1" "stable-baselines3>=2.0" "gymnasium>=0.29" pyyaml
import os, sys, sumo, torch
os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
os.environ["PATH"] = os.path.join(os.environ["SUMO_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["MPLBACKEND"] = "Agg"
os.chdir(WORK)
!python run.py check
WORKERS = os.cpu_count()
print(f"\nCPU cores: {WORKERS} -> SUMO workers {WORKERS};  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (DeepONet trains on CPU)'}")

In [ ]:
# 4. (first session) one SUMO episode on the 60 km/h road, ~1 min
!python run.py simulate --policy u=0.5 --out runs/simulation/colab_check.npz | tail -5
import glob, sumolib
net = sorted(glob.glob("data/networks/simulate/**/*.net.xml", recursive=True), key=os.path.getmtime)[-1]
n = sumolib.net.readNet(net)
print(f"{net}: ramp {n.getEdge('ramp').getSpeed() * 3.6:.0f} km/h, acceleration lane {n.getEdge('highway_accel').getSpeed() * 3.6:.0f} km/h")

## Step 1: screening and capacity check (decision point)

`run.py e0` runs the 404 E0 screening episodes (they also become part of the initial dataset) and compares the merge
capacity with the 120 km/h study (`configs/reference/e0_ramp120kmh.json`): for each profile, the lowest constant meter
rate at which the merge breaks down.

- **keep**: continue with cell 6.
- **rescale** (mean shift of one grid step, 120 veh/h, or more): stop here. The storage-mandatory threshold, the
  feedforward capacity range and the demand ceilings would have to be recalibrated first. `run.py pipeline` also refuses
  to generate data in that case unless given `--ignore-capacity-check`.

In [ ]:
# 5. E0 + capacity check (~404 SUMO episodes: ~25 min on 8 cores); reused if already done
!python run.py e0 --workers {WORKERS}
import json
check = json.load(open("runs/study/m14/e0_capacity_comparison.json"))
print("\nVERDICT:", check["verdict"], f"(mean shift {check['mean_shift_vph']:+.0f} veh/h)")

## Step 2: the study

Cell 6 starts `run.py pipeline --parallel --recover-interrupted` as a background process: the data stage, then three
branches at once ([DeepONet -> surrogate-PPO], direct SUMO-PPO, baselines), then the final evaluation, the tables and
the figures. Logs: `runs/logs/pipeline_driver.log` and `runs/logs/pipeline_{surrogate,direct,baselines}.log`.

**After a lost session** rerun cells 1-3, then cell 6. Cell 6 refuses to start a second driver while one is running.

In [ ]:
# 6. Launch or resume the pipeline in the background
import os, sys, subprocess, json
os.makedirs("runs/logs", exist_ok=True)
pid_file = "runs/logs/pipeline_driver.pid"

def driver_alive():
    if not os.path.exists(pid_file):
        return False
    try:
        pid = int(open(pid_file).read().strip())
        os.kill(pid, 0)
        state = [l for l in open(f"/proc/{pid}/status") if l.startswith("State:")][0]
        return "Z" not in state   # a finished driver stays a zombie until reaped
    except (ProcessLookupError, ValueError, FileNotFoundError, IndexError):
        return False              # a pid file from a previous runtime

if driver_alive():
    print("A pipeline driver is already running in this runtime; see cell 7.")
else:
    cmd = [sys.executable, "run.py", "pipeline", "--parallel", "--recover-interrupted", "--workers", str(WORKERS),
           "--seed", str(SEED), "--rounds", str(ROUNDS), "--budgets", *map(str, BUDGETS)]
    for flag, values in ALINEA_GRID.items():
        cmd += [f"--{flag}", *map(str, values)]
    log = open("runs/logs/pipeline_driver.log", "a")
    log.write("\n\n===== launch " + " ".join(cmd) + "\n"); log.flush()
    proc = subprocess.Popen(cmd, cwd=WORK, stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=dict(os.environ))
    open(pid_file, "w").write(str(proc.pid))
    print("pipeline driver started, pid", proc.pid)

In [ ]:
# 7. Status (rerun any time)
import os, glob, json
def tail(path, n=4):
    if os.path.exists(path):
        lines = open(path, errors="replace").read().splitlines()
        print(f"--- {path}"); print("\n".join(lines[-n:]))

print("driver running:", driver_alive() if "driver_alive" in globals() else "unknown (run cell 6's definitions)")
tail("runs/logs/pipeline_driver.log", 6)
for branch in ("surrogate", "direct", "baselines"):
    tail(f"runs/logs/pipeline_{branch}.log", 3)

ens = "runs/deeponet/round0"
print("\nDeepONet members done:", len(glob.glob(f"{ens}/member_*/completed.json")), "/ 5", "(ensemble complete)" if os.path.exists(f"{ens}/manifest.json") else "")
rounds = "runs/aggregation/m14_s0/rounds.json"
if os.path.exists(rounds):
    for r in json.load(open(rounds)):
        print(f"  round {r['round']}: SUMO V {r['best_sumo_val']:.1f}, cumulative {r['cumulative_ee']} SUMO episodes")
if os.path.exists("runs/aggregation/m14_s0/study.json"):
    print("  aggregation loop finished")
for run in sorted(glob.glob("runs/study/m14/direct_ppo_*ee_s0")):
    ckpts = glob.glob(f"{run}/checkpoints/*_steps.zip")
    last = max((int(p.split("_")[-2]) for p in ckpts), default=0)
    total = int(run.split("_")[-2][:-2]) * 120
    print(f"direct PPO {os.path.basename(run)}: {last} / {total} steps ({100 * last / total:.0f} %)",
          "- finished" if os.path.exists(f"{run}/final_model.zip") else "")
tuning_path = "runs/study/m14/alinea_tuning.json"
print("baselines tuned:", os.path.exists(tuning_path))
if os.path.exists(tuning_path):
    tuning = json.load(open(tuning_path))
    for name in ("best_pure_alinea", "best_pi_alinea"):
        edges = tuning.get("edge_check", {}).get(name, {})
        print(f"  {name}: {tuning.get(name)}", f"-- ON THE GRID EDGE {edges}: widen the grid and retune" if edges else "(inside the grid)")
print("final evaluation files:", len(glob.glob("runs/study/m14/eval/*.summary.json")))
print("tables:", os.path.exists("runs/study/m14/tables/tables.md"))
errors = [p for p in glob.glob("runs/logs/*.log") if "Traceback" in open(p, errors="replace").read()]
print("logs with a Traceback:", errors or "none")

In [ ]:
# 8. Tables I and II, headline reductions
from IPython.display import Markdown, display
path = "runs/study/m14/tables/tables.md"
display(Markdown(open(path).read()) if os.path.exists(path) else Markdown("not built yet (cell 7 shows progress)"))

In [ ]:
# 9. Archive of the results (small files only) next to WORK on Drive
import shutil, datetime, tempfile
keep = ["runs/study/m14/tables", "runs/study/m14/arms.json", "runs/study/m14/alinea_tuning.json",
        "runs/study/m14/e0.json", "runs/study/m14/e0_capacity_comparison.json", "runs/aggregation/m14_s0/rounds.json",
        "runs/aggregation/m14_s0/study.json", "runs/ledger", "runs/logs", "runs/commands.jsonl", "reports", "CODE_VERSION.txt"]
keep += glob.glob("runs/study/m14/eval/*.jsonl") + glob.glob("runs/study/m14/eval/*.summary.json")
stage = tempfile.mkdtemp()
for item in keep:
    if os.path.isdir(item):
        shutil.copytree(item, os.path.join(stage, item), dirs_exist_ok=True)
    elif os.path.exists(item):
        os.makedirs(os.path.dirname(os.path.join(stage, item)) or stage, exist_ok=True)
        shutil.copy2(item, os.path.join(stage, item))
name = f"{WORK}_results_{datetime.datetime.now():%Y%m%d_%H%M}"
print(shutil.make_archive(name, "zip", stage))